<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">چند <bdi dir="ltr">Head</bdi>، یک خروجی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چگونه چند مسیر <bdi dir="ltr">Attention</bdi> دوباره به یک نمایش <bdi dir="ltr">C</bdi>تایی برمی‌گردند؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/35-split-heads.html"><bdi dir="ltr">35-split-heads</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html"><bdi dir="ltr">36-merge-heads</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">خروجی <bdi dir="ltr">Attention</bdi> واقعی پروژه را با محاسبهٔ خودمان مقایسه می‌کنیم. مرحله‌ها را با همان وزن‌ها بازسازی می‌کنیم تا تفاوتی از وزن‌های تصادفی وارد مقایسه نشود. <bdi dir="ltr">B</bdi> تعداد نمونه، <bdi dir="ltr">T</bdi> طول، <bdi dir="ltr">C</bdi> تعداد ویژگی، <bdi dir="ltr">H</bdi> تعداد <bdi dir="ltr">Head</bdi> و <bdi dir="ltr">D</bdi>=<bdi dir="ltr">C/H</bdi> است. قبل از اجرا، شکل خروجی <bdi dir="ltr">Layer</bdi> مشترک <bdi dir="ltr">QKV</bdi> و جدول وزن‌های <bdi dir="ltr">Attention</bdi> را بنویسید.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
B,T,C,H = 2,5,12,3
D = C//H
config = ModelConfig(vocab_size=12,context_length=8,embedding_dim=C,
                     num_heads=H,num_layers=1,dropout=0.)
attention = CausalSelfAttention(config).eval()
x = torch.randn(B,T,C)
trace = {}
with torch.no_grad():
    output, weights = attention(x, return_weights=True, trace=trace)
    combined_qkv = attention.qkv(x)
    q_flat,k_flat,v_flat = combined_qkv.chunk(3,dim=-1)
    for name, flat in zip(("q","k","v"),(q_flat,k_flat,v_flat)):
        heads = flat.reshape(B,T,H,D).transpose(1,2)
        torch.testing.assert_close(heads,trace[name])
    per_head = weights @ trace["v"]
    torch.testing.assert_close(per_head,trace["weighted_values"])
    merged = per_head.transpose(1,2).contiguous().view(B,T,C)
    projected = attention.output(merged)
    torch.testing.assert_close(projected,output)
for name, value in [("X",x),("QKV",combined_qkv),("Q before split",q_flat),
                    ("Q heads",trace["q"]),("weights",weights),
                    ("weighted V",per_head),("merged",merged),("projected",output)]:
    inspect(name,value)


In [ ]:
fig, axes = plt.subplots(1,H,figsize=(3*H,3),squeeze=False)
for head, ax in enumerate(axes[0]):
    ax.imshow(weights[0,head],vmin=0,vmax=1,cmap="Blues")
    ax.set(title=f"Head {head}",xlabel="Key",ylabel="Query")
plt.tight_layout()
plt.show()
torch.testing.assert_close(weights.sum(-1),torch.ones(B,H,T))
assert torch.count_nonzero(weights.triu(1)) == 0
try:
    ModelConfig(vocab_size=12,embedding_dim=10,num_heads=3)
except ValueError as error:
    print("Expected invalid C/H:",error)
else:
    raise AssertionError("C must be divisible by H")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> با <bdi dir="ltr">C</bdi>=12، <bdi dir="ltr">H</bdi> را به ۱ و سپس ۴ تغییر دهید و کل دفتر را از نو اجرا کنید. <bdi dir="ltr">D</bdi> و شکل جدول وزن چه می‌شوند؟ شمار <bdi dir="ltr">Parameter</bdi>های <bdi dir="ltr">QKV</bdi> و <bdi dir="ltr">Output projection</bdi> را مقایسه کنید. انتظار نداریم <bdi dir="ltr">Head</bdi>های تصادفی از پیش نقش‌های زبانی معنادار داشته باشند. برابرشدن <bdi dir="ltr">Shape</bdi> کافی نبود؛ به همین دلیل مقدارهای ادغام و <bdi dir="ltr">Projection</bdi> را هم آزمودیم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: سهم هر <bdi dir="ltr">Head</bdi> پس از <bdi dir="ltr">Projection</bdi> نهایی</h2>
<p style="text-align:right">خروجی <bdi dir="ltr">Multi-Head</bdi> را به سهم‌های خطی <bdi dir="ltr">Head</bdi>ها تجزیه کنید، بدون چندبار افزودن <bdi dir="ltr">Bias</bdi>. پیش‌نیاز: تقسیم، ادغام و <bdi dir="ltr">Output projection</bdi> همین دفتر را بشناسید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right"><bdi dir="ltr">Projection</bdi> نهایی روی ویژگی‌های چسبانده‌شده عمل می‌کند. آیا می‌توان سهم هر قطعه را جدا ضرب کرد و جمع زد؟ <bdi dir="ltr">Bias</bdi> باید چند بار اضافه شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
attention = CausalSelfAttention(ModelConfig(9,6,6,2,1,0.)).eval()
with torch.no_grad(): attention.output.bias.fill_(0.4)
x = torch.randn(1,3,6)
trace = {}
with torch.no_grad(): output = attention(x,trace=trace)
attended = trace['weighted_values']
print('per-head tensor:',attended.shape)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">head_contributions(attention,attended)</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,C)</code> برگرداند. برای هر <bdi dir="ltr">Head</bdi> فقط ستون‌های متناظر آن را از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">attention.output.weight</code> بگیرید و سهم بدون <bdi dir="ltr">Bias</bdi> را حساب کنید. مجموع روی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H</code> به‌اضافهٔ یک <bdi dir="ltr">Bias</bdi> باید خروجی اصلی را بسازد.</p>
</div>

In [ ]:
def head_contributions(attention, attended):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = head_contributions(attention,attended)
    if result is None: return False
    assert result.shape == (1,2,3,6)
    torch.testing.assert_close(result.sum(1)+attention.output.bias,output)
    other = CausalSelfAttention(ModelConfig(9,6,6,3,1,0.)).eval()
    a = torch.randn(2,3,4,2)
    pieces = head_contributions(other,a)
    merged = a.transpose(1,2).reshape(2,4,6)
    torch.testing.assert_close(pieces.sum(1)+other.output.bias,other.output(merged))
    for h in range(3):
        torch.testing.assert_close(pieces[:,h],a[:,h]@other.output.weight[:,2*h:2*h+2].T)
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط خروجی <bdi dir="ltr">Head</bdi> صفر را پیش از ادغام صفر کنید؛ وزن‌ها ثابت‌اند. این حذف مسیر عددی است، نه شاهدی دربارهٔ نقش زبانی یا اهمیت آموختهٔ آن <bdi dir="ltr">Head</bdi>.</p>
</div>

In [ ]:
removed = attended.clone(); removed[:,0] = 0
with torch.no_grad():
    changed_output = attention.output(removed.transpose(1,2).reshape(1,3,6))
print('head-zero removal effect:',output-changed_output)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب <bdi dir="ltr">Bias</bdi> را به سهم تک‌تک <bdi dir="ltr">Head</bdi>ها اضافه می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">combine_contributions(pieces,bias)</code> باید سهم‌ها را روی محور <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H</code> جمع و <bdi dir="ltr">Bias</bdi> را یک بار اضافه کند.</p>
</div>

In [ ]:
pieces = torch.ones(1,2,3,6)
bias = torch.full((6,),0.4)
print('wrong repeated bias:',(pieces+bias).sum(1)[0,0])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def combine_contributions(pieces, bias):
    # TODO
    return None

In [ ]:
def test_repair():
    result = combine_contributions(pieces,bias)
    if result is None: return False
    torch.testing.assert_close(result,torch.full((1,3,6),2.4))
    a = torch.arange(24.).reshape(1,3,2,4)
    b = torch.tensor([1.,2.,3.,4.])
    torch.testing.assert_close(combine_contributions(a,b),a[:,0]+a[:,1]+a[:,2]+b)
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">این تجزیه دقیقاً از وزن <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">output</code> در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CausalSelfAttention</code> استفاده می‌کند. جمع سهم‌ها یک خاصیت <bdi dir="ltr">Linear</bdi> است؛ وزن <bdi dir="ltr">Attention</bdi> هر <bdi dir="ltr">Head</bdi> قبلاً در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">attended</code> مصرف شده است.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا نقش <bdi dir="ltr">Output projection</bdi> بیشتر از کنار هم چیدن <bdi dir="ltr">Head</bdi>هاست، حتی وقتی می‌توان آن را به سهم‌های خطی تجزیه کرد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-09_multi_head.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>